# Kaggriculture：结合生产路线与有界市场优化的农场代理

**最终策略讲解、实现说明与可复现构建**

本笔记只介绍一个完整的农场代理：如何安排生产与交付，把可兑现的库存变成现金，保护持续生产所需资源，并在有限计算量内优化市场成交顺序。

代理结合了有明确署名的公开生产控制器和成交模型，以及库存销售接入、资金保护、同回合搜索缓存、纯卖单顺序优化和诊断统计。公开生产路线与继承的市场模型不被宣称为本地原创。

**适合读者：** 熟悉 Python 列表、字典和函数的学习者。  
**学习目标：** 理解行动流程、解释卖单顺序为何重要、识别搜索边界，并恢复经过核验的独立代理归档。  
**运行条件：** CPU 即可，默认离线，不需要 API 密钥。

从头运行会执行教学例子、完整性检查并恢复发布文件，不会自动安装依赖、重跑全部历史对局、提交比赛或公开发布。


## 阅读目录

1. [代理的目标](#objective)
2. [生产与交付](#production)
3. [库存和资金保护](#safeguards)
4. [为什么成交顺序重要](#market-example)
5. [有界市场优化](#optimization)
6. [实现与诊断](#implementation)
7. [验证与限制](#validation)
8. [练习](#exercise)
9. [构建与核验](#build)
10. [来源与发布](#attribution)


In [1]:
# 计算仅使用标准库；IPython 只负责展示易读表格。
import ast
import base64
import gzip
import hashlib
import io
import json
import tarfile
from collections import Counter
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    def Markdown(value):
        return value
    def display(value):
        print(value)

def show_table(headers, rows):
    def escape(value):
        return str(value).replace('|', chr(92) + '|').replace(chr(10), ' ')
    lines = ['| ' + ' | '.join(map(escape, headers)) + ' |',
             '| ' + ' | '.join('---' for _ in headers) + ' |']
    lines += ['| ' + ' | '.join(map(escape, row)) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))

print('运行模式：离线教学、完整性检查与文件恢复。')
print('历史游戏引擎：kaggle-environments 1.32.7。')


运行模式：离线教学、完整性检查与文件恢复。
历史游戏引擎：kaggle-environments 1.32.7。


<a id="objective"></a>
## 1. 代理的目标

Kaggriculture 是资源管理游戏，不是预测 CSV 的表格任务。代理每次读取观测，返回农夫、工人和市场订单的行动；生产、采集、运输与销售必须协同。

经营目标是在保护持续生产资源的前提下赚取现金。市场搜索中的“相对成交收入”只是局部评分，不等于平台评分。

下面展示概念流程，而非逐函数调用栈：

```text
当前观测
    → 根据商店与农场状态安排生产及工人行动
    → 必要时执行末期有界采集与运输规划
    → 投影仓库可用库存
    → 构造受保护的采购与销售队列
    → 约束条件下搜索成交顺序
    → 返回行动并同步诊断状态
```

比赛策略只使用当前可获得的信息，不读取未来对手行动、未揭示商店安排或对手源码。


<a id="production"></a>
## 2. 生产与交付

生产控制器结合继承的路线与状态规则，协调作物、动物、饲料、投入资源和工人移动。已揭示商店参与路线选择，而不是在所有场景都种同一种作物。

| 职责 | 代理的处理 | 为什么重要 |
| --- | --- | --- |
| 选择路线 | 根据已揭示商店与实时农场状态选择生产方案 | 不同需求结构适合不同生产计划 |
| 协调工人 | 安排种植、照料、采集与交付 | 产品未被采集和送回仓库，就不能兑现 |
| 保护经营资源 | 保留饲料与已承诺的投入资源 | 把所有库存卖光可能中断后续生产 |
| 末期物理收尾 | 最后七次回调中按需使用有界采集与交付规划 | 留在工人背包或地块上的货物可能无法变现 |

历史配置包含 720 个状态步，末期规划覆盖回调 712—718。这是短窗口的物理行动规划，不是整局搜索。生产路线与需求的匹配仍然是关键限制：成交顺序不能弥补很大的产销错配。


<a id="safeguards"></a>
## 3. 库存与资金保护

### 3.1 卖出可交付库存，而不是假想产量

添加现金产品卖单前，控制器按照计划中的单位行动投影仓库状态，计入当前回合能够完成的交付，并识别被取货指令预留的产品。

额外销售检查可用数量和当前报价，再用“对手同时销售同量产品”的保守模型限制数量。模型报价低于 2 时停止增加；这一保护不缩减原先已安排的销售数量。

该层不会把饲料、动物和肥料当作可随意清空的现金产品。新增卖单也要遵守市场订单容量。

### 3.2 保护生产需要的采购

只有资金和订单依赖检查通过，受保护的小麦采购才可能排到固定价格种子单之前。保守成本上界会预留相关采购资金，不依赖尚未实现的销售收入来融资。

同品种买卖循环、雇工、土地等行动具有额外状态依赖，不能按简化模型任意重排。最后的纯卖单优化只在连续 SELL 块内工作，不跨越固定采购位置。


<a id="market-example"></a>
## 4. 为什么成交顺序重要

成交模型考虑订单位置和逐单位交错报价。一批产品先进入市场，可能改变后续同类产品的报价，因此相同销售数量也可能因排列不同而产生不同收入。

下方采用人为简化的定价：`max(1, 基础价 - 10 × 库存)`，只支持 SELL，不考虑生产、采购、容量和商店消耗。它是**教学例子，不是官方价格公式或代理性能实验**。

例子保留一个关键细节：每一轮先给双方报价，再提交双方成交，不能先让一方成交后再给另一方报价。


In [2]:
def teaching_market(own_orders, rival_orders):
    inventory = {'MILK': 0, 'WOOL': 0}
    base_price = {'MILK': 100, 'WOOL': 60}
    revenue = [0, 0]
    queues = [own_orders, rival_orders]
    for slot in range(max(map(len, queues))):
        active = [list(q[slot]) if slot < len(q) else None for q in queues]
        while any(order and order[2] > 0 for order in active):
            # 先给双方报价，再提交双方成交。
            quotes = [max(1, base_price[o[1]] - 10 * inventory[o[1]])
                      if o and o[2] > 0 else None for o in active]
            for seat, quote in enumerate(quotes):
                if quote is None:
                    continue
                item = active[seat][1]
                revenue[seat] += quote
                if quote > 1:
                    inventory[item] += 1
                active[seat][2] -= 1
    return tuple(revenue)

original_orders = [['SELL', 'WOOL', 1], ['SELL', 'MILK', 4]]
reordered_orders = list(reversed(original_orders))
before = teaching_market(original_orders, original_orders)
after = teaching_market(reordered_orders, original_orders)
assert before == (340, 340) and after == (390, 240)
assert Counter(map(tuple, original_orders)) == Counter(map(tuple, reordered_orders))
show_table(['我方队列', '我方收入', '对手收入', '收入差'], [
    ['WOOL → MILK', *before, before[0] - before[1]],
    ['MILK → WOOL', *after, after[0] - after[1]],
])


| 我方队列 | 我方收入 | 对手收入 | 收入差 |
| --- | --- | --- | --- |
| WOOL → MILK | 340 | 340 | 0 |
| MILK → WOOL | 390 | 240 | 150 |

例子没有改变产品数量，相对收入却从 0 变成了 +150。但这**不说明牛奶永远应排第一**。真实选择取决于官方价格曲线、库存、单量、采购位置以及对手行为。


<a id="optimization"></a>
## 5. 有界市场优化

代理接入现金产品销售，并使用继承的逐单位成交模型评估顺序。纯卖单收尾假设对手使用镜像队列，以估计的相对收入评分：

```text
评分 = 预计我方收入 - 预计对手收入
只有评分提升超过 0.5，才接受候选排列
```

### 5.1 连续卖单块

最后一层考虑连续的两至四条 SELL。四条订单每次最多有 `4! = 24` 种排列，保持销售数量、单位指令和固定采购位置不变。

外层最多执行四次，行动不再变化或完整行动键重复时停止。如果**任意**连续 SELL 块超过四单，整个新增收尾层直接返回继承行动，而不是只跳过那一个大块。

控制器还包含受保护的联合队列优化和固定订单处理。因此“四次”指最后的纯卖单收尾层，不代表整个继承控制器的总计算量。

### 5.2 缓存与状态一致性

确定性搜索只缓存当前观测与行动上下文。单位指令也进入缓存键，因为它们影响可交付库存。返回重排后的行动时，同步记录实际返回的队列，避免预测器记住已被替换的旧行动。

镜像模型只是近似，并不知道对手真正要做什么。模型评分更高，不保证真实收入更高或一定获胜。


<a id="implementation"></a>
## 6. 实现与诊断

下方摘录发布源码中的真实函数：它封装生产与市场控制器，调用既有成交模型，约束搜索规模，并同步实际返回的行动。

摘录依赖完整代理的辅助函数和状态，适合阅读，不能单独当作独立提交运行。


```python
def sales_closure_submission_agent(observation,configuration=None):
    if int(observation['step'])==0:_P19_REPORT.update(calls=0,changed=0,errors=0)
    _P19_REPORT['calls']+=1
    action=_P19_BASE(observation,configuration)
    try:
        # 每个连续卖单块最多四单，保持数量和单位指令不变。
        run=0
        for order in action.get('market',[]):
            run=run+1 if order and order[0]=='SELL' else 0
            if run>4:return action
        result=action;seen=set()
        for _ in range(4):
            key=_s834_key(result)
            if key in seen:break
            seen.add(key)
            revised=_v44y_reorder(observation,result)
            if revised==result:break
            result=revised
        if result!=action:
            _P19_REPORT['changed']+=1
            state=_RACE_STATE.get(int(observation['player']))
            if state is not None and state.get('step')==int(observation['step']):state['prev_action']=result
        return result
    except Exception:
        _P19_REPORT['errors']+=1
        return action
```


公开入口 `submission_agent` 返回行动，并汇总回调数、库存销售变化、受保护采购调整、订单重排、资金保护触发和错误计数。诊断数据不参与选择行动。

纯卖单封装发生异常时，会回退到继承行动并增加错误计数。这是可用性保护，不代表对手模型已经正确。


<a id="validation"></a>
## 7. 验证与限制

保存的完整对局检查采用 `kaggle-environments==1.32.7`，每场代理接受 719 次回调。下方记录属于最终决策策略的历史执行检查，不是本笔记新跑出的对局，也不能估计排行榜实力。

发布副本调整了中文注释、说明字符串与中性的公开入口名称，因此有自己的文件哈希。决策代码在这些明确变动归一化后比较语法结构，同观测行动一致性另行核验。


In [3]:
HISTORICAL_CHECKS = json.loads('{"engine_version": "1.32.7", "games": 6, "wins": 4, "ties": 0, "losses": 2, "errors": 0, "calls_per_game": 719, "description": "最终决策策略的历史完整对局执行检查，不是新进行的排行榜评估。", "original_report_sha256": "898a11c7b32fe0ede3794934f7e0217ef85e2c7e67e067137b97b9c070c1dc4e", "results": [{"seed": 105501, "seat": 0, "margin": -202.0, "candidate_calls": 719, "done": true}, {"seed": 105501, "seat": 1, "margin": 1080.0, "candidate_calls": 719, "done": true}, {"seed": 105502, "seat": 0, "margin": 76.0, "candidate_calls": 719, "done": true}, {"seed": 105502, "seat": 1, "margin": -76.0, "candidate_calls": 719, "done": true}, {"seed": 105503, "seat": 0, "margin": 167.0, "candidate_calls": 719, "done": true}, {"seed": 105503, "seat": 1, "margin": 167.0, "candidate_calls": 719, "done": true}]}')

rows = HISTORICAL_CHECKS['results']
assert len(rows) == 6
assert all(r['done'] and r['candidate_calls'] == 719 for r in rows)
assert HISTORICAL_CHECKS['errors'] == 0
show_table(['检查项目', '保存的结果'], [
    ['完整对局执行检查', str(len(rows)) + ' 场完整对局'],
    ['每场回调次数', '719'],
    ['已记录运行错误', '0'],
    ['历史引擎', HISTORICAL_CHECKS['engine_version']],
])
print('这些是保存的执行检查，本单元没有启动比赛对局。')


| 检查项目 | 保存的结果 |
| --- | --- |
| 完整对局执行检查 | 6 场完整对局 |
| 每场回调次数 | 719 |
| 已记录运行错误 | 0 |
| 历史引擎 | 1.32.7 |

这些是保存的执行检查，本单元没有启动比赛对局。


### 重要限制

- 生产路线与需求适配仍然是基本约束。
- 镜像模型可能误判对手的成交时机和数量。
- 有界局部搜索不能证明行动或农场经营策略全局最优。
- 单台机器上的耗时不能作为跨硬件保证。
- 正常执行、语法一致和行动一致回答的是不同问题；任何一项都不能独立证明竞技实力。

本笔记不承诺奖牌、最终名次或某个特定平台评分。


<a id="exercise"></a>
## 8. 练习：换一个对手队列，先预测再运行

把教学例子中的对手顺序改为 `MILK → WOOL`。运行前先预测我方两种顺序各自得到的收入，将两组预测填入 `expected`，再取消断言的注释进行检查。

另一个问题：六条订单共有多少种排列？为什么扩大搜索不能自动纠正错误的对手假设？


In [4]:
rival_queue = [['SELL', 'MILK', 4], ['SELL', 'WOOL', 1]]
expected = []  # 检查前先填入两组（我方收入，对手收入）预测。
actual = [teaching_market(original_orders, rival_queue),
          teaching_market(reordered_orders, rival_queue)]
# assert expected == actual
print('教学模型得到的收入对：', actual)


教学模型得到的收入对： [(240, 390), (340, 340)]


<details>
<summary>展开参考答案</summary>

两组收入为 `[(240, 390), (340, 340)]`。六条订单有 `6! = 720` 种排列，而四条只有 `4! = 24`。增加枚举会增加计算量，但不会自动修正对手队列或单量假设。

</details>


<a id="build"></a>
## 9. 构建与核验代理

Kaggle 自包含版在折叠输入单元中携带压缩归档；GitHub 轻量版读取随仓库提供的同一归档。两版恢复相同的源码、LICENSE 和 NOTICE 文件。

解码后依次检查 SHA-256、归档成员白名单、语法编译和公开入口。构建不会执行恢复出的比赛代理，也不会提交比赛。许可证与上游版权声明保持原文，其余学习说明使用中文。


In [5]:
# GitHub 轻量版读取随目录提供的归档。
roots = [Path.cwd(), *Path.cwd().parents]
artifact = next((p / 'artifacts/submission.tar.gz' for p in roots
                 if (p / 'artifacts/submission.tar.gz').is_file()), None)
if artifact is None:
    raise FileNotFoundError('请保留完整发布目录，或使用 Kaggle 自包含版笔记。')
packed_bytes = artifact.read_bytes()
print('归档已载入，字节数：', len(packed_bytes))


归档已载入，字节数： 666133


In [6]:
EXPECTED_SOURCE_SHA = 'bb1dda21b5b034dd56e1b28c393e2b0b1e7d44052b45852296da0e6aab2047b3'
EXPECTED_ARCHIVE_SHA = 'd5e0e102634fe28866f12ff68f288b88888d93910c9dc17c9626b4a7e3b53a3c'

assert hashlib.sha256(packed_bytes).hexdigest() == EXPECTED_ARCHIVE_SHA
allowed = ['main.py', 'LICENSE.txt', 'NOTICE.txt']
with tarfile.open(fileobj=io.BytesIO(packed_bytes), mode='r:gz') as archive:
    assert archive.getnames() == allowed
    assert all(member.isfile() for member in archive.getmembers())
    restored = {name: archive.extractfile(name).read() for name in allowed}
source_bytes = restored['main.py']
assert hashlib.sha256(source_bytes).hexdigest() == EXPECTED_SOURCE_SHA
compile(source_bytes, 'main.py', 'exec')
tree = ast.parse(source_bytes)
entry_name = [n.name for n in tree.body if isinstance(n, ast.FunctionDef)][-1]
assert entry_name == 'submission_agent'
assert b'Apache License' in restored['LICENSE.txt']
print('通过：归档哈希、源码哈希、成员白名单、语法与公开入口。')
print('公开入口：', entry_name)


通过：归档哈希、源码哈希、成员白名单、语法与公开入口。
公开入口： submission_agent


In [7]:
# 写入独立输出目录；已有不同内容时拒绝覆盖。
kaggle_work = Path('/kaggle/working')
output_dir = (kaggle_work if kaggle_work.is_dir() else Path.cwd()) / 'strategy_outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_files = dict(restored, **{'submission.tar.gz': packed_bytes})
for name, data in output_files.items():
    target = output_dir / name
    if target.exists() and target.read_bytes() != data:
        raise FileExistsError('拒绝覆盖已有的不同内容：' + name)
    if not target.exists():
        target.write_bytes(data)
show_table(['输出文件', '字节数'], [[name, len(data)] for name, data in output_files.items()])
print('输出目录：strategy_outputs。没有自动提交或发布。')


| 输出文件 | 字节数 |
| --- | --- |
| main.py | 1211597 |
| LICENSE.txt | 11358 |
| NOTICE.txt | 6109 |
| submission.tar.gz | 666133 |

输出目录：strategy_outputs。没有自动提交或发布。


### 可选：运行一场完整功能检查

发布目录包含独立工具 `run_match.py`。在单独的 Python 3.11 环境中显式安装历史引擎后，可执行：

```bash
python -m pip install -r requirements-engine.txt
python run_match.py --seed 20261015 --opponent random
```

这只是对内置对手的功能检查，不是竞技实力评估。本笔记没有宣称上述建议种子已经运行或取得某种结果。默认构建不会安装或调用比赛引擎。


<a id="attribution"></a>
## 10. 来源与发布

最终策略整合了已有公开控制器与成交模型，不把继承的生产路线和市场模型说成本地原创。

| 来源 | 与本代理的关系 |
| --- | --- |
| [Master Hybrid](https://www.kaggle.com/code/haideptry/the-2965-master-hybrid-engine) | 直接公开控制器基础 |
| [Demand-Preserving Turn Sale Timing](https://www.kaggle.com/code/tetsutani/demand-preserving-turn-sale-timing) | 库存销售时机与来源链 |
| [Your Market List Is an Order Book](https://www.kaggle.com/code/shiiin9/your-market-list-is-an-order-book) | 市场控制器来源链 |
| [Kaggle 官方环境](https://github.com/Kaggle/kaggle-environments) | 历史游戏语义与物理行动定义 |

完整来源还包括 Ahmed Berat Ozer、Yusuke Hayashi、Thomas Tschinkel、aurax7、Dmitrii Gluzdov，以及源码和 NOTICE 中列明的其他贡献者。归档保留 Apache-2.0 许可证及上游版权声明原文。

上游 NOTICE 指出部分路线数据缺少完整的逐对局来源清单。本发布材料没有补造缺失来源，也不声称能从原始回放重训全部继承路线。

Kaggle 上先发布 Notebook，再在 Discussion 摘要中加入笔记链接。GitHub 上传解压后的目录内容，保留源码、归档与许可文件。不会自动代为发布。

**总结：** 生产路线形成库存，交付让库存可以出售，资金保护维持经营，有界订单搜索则在明确但近似的对手模型下改善成交执行。
